# Gestión de pedidos con openpyxl

Notebook didáctico para trabajar directamente con `BDD/BaseDatosProyecto.xlsx` sin Pandas.

El recorrido está ordenado desde la preparación hasta la exportación:

1. Localizar y copiar el Excel.
2. Abrir el libro y seleccionar la hoja.
3. Leer y normalizar cabeceras.
4. Crear la columna `Total` si no existe.
5. Calcular `Precio unidad × Cantidad × (1 − Descuento)`.
6. Leer filas como diccionarios.
7. Consultar mediante filtros.
8. Crear, leer, actualizar y eliminar registros.
9. Usar formularios interactivos opcionales.
10. Crear resúmenes, una hoja de resumen y un gráfico Excel.
11. Verificar y guardar el resultado.

> El archivo original no se modifica. El notebook trabaja con una copia llamada `BaseDatosProyecto_trabajo.xlsx`.


<a id="indice-contenido"></a>

## Índice de contenido

### Preparación del entorno y del archivo

1. [Preparar el entorno e instalar openpyxl e ipywidgets](#paso-01)
2. [Importar los módulos para Excel, fechas, rutas e interactividad](#paso-02)
3. [Definir el visor HTML para mostrar registros sin Pandas](#paso-03)
4. [Localizar y validar la ruta del archivo Excel original](#paso-04)
5. [Crear una copia de trabajo para proteger el archivo original](#paso-05)
6. [Cargar el libro y seleccionar la hoja de gestión de pedidos](#paso-06)

### Estructura, limpieza y cálculo de datos

7. [Inspeccionar las cabeceras originales y detectar caracteres ocultos](#paso-07)
8. [Limpiar saltos de línea y espacios repetidos en las cabeceras](#paso-08)
9. [Verificar que existan las nueve columnas obligatorias](#paso-09)
10. [Crear el mapa dinámico entre nombres y números de columna](#paso-10)
11. [Añadir o reutilizar la columna Total al final de la hoja](#paso-11)
12. [Definir las validaciones y la fórmula del importe total](#paso-12)
13. [Calcular y guardar el Total de todos los pedidos existentes](#paso-13)
14. [Guardar la estructura y los totales en la copia de trabajo](#paso-14)
15. [Convertir las filas de Excel en registros identificados por fila física](#paso-15)

### Consultas y filtros

16. [Construir una función reutilizable de filtros combinados](#paso-16)
17. [Ejecutar una consulta de ejemplo y revisar sus coincidencias](#paso-17)
18. [Explorar pedidos con un panel de filtros opcionales](#paso-18)

### Operaciones CRUD e interfaces

19. [Validar textos, fechas y números antes de ejecutar el CRUD](#paso-19)
20. [Implementar la creación y consulta con Total automático](#paso-20)
21. [Implementar la actualización protegida y la eliminación de pedidos](#paso-21)
22. [Probar las operaciones CRUD mediante ejemplos manuales](#paso-22)
23. [Gestionar pedidos desde una consola con búsqueda previa](#paso-23)
24. [Gestionar pedidos con ipywidgets y validación en tiempo real](#paso-24)

### Resumen, gráfico y exportación

25. [Agrupar pedidos por país y conciliar cantidades y totales](#paso-25)
26. [Crear la hoja Resumen y el gráfico de totales por país](#paso-26)
27. [Guardar en Excel todos los cambios realizados en memoria](#paso-27)
28. [Reabrir el archivo y ejecutar las comprobaciones básicas](#paso-28)
29. [Mostrar el resultado consolidado final en formato Markdown](#paso-29)

Los enlaces utilizan identificadores HTML estables (`paso-01` a `paso-29`) y continúan funcionando aunque se cambie el texto de los títulos.


<a id="paso-01"></a>

## Paso 1. Preparar el entorno e instalar openpyxl e ipywidgets

Solo `openpyxl` es obligatorio para manipular Excel. `ipywidgets` se utiliza para los formularios interactivos.


In [32]:
# %pip install openpyxl ipywidgets


<a id="paso-02"></a>

## Paso 2. Importar los módulos para Excel, fechas, rutas e interactividad

No se importa Pandas.


In [33]:
from pathlib import Path
from shutil import copy2
from datetime import datetime, date
from collections import defaultdict
from html import escape
import re

from openpyxl import load_workbook, Workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from IPython.display import display, HTML, Markdown, clear_output

try:
    import ipywidgets as widgets
    WIDGETS_DISPONIBLES = True
except ImportError:
    widgets = None
    WIDGETS_DISPONIBLES = False
    print("ipywidgets no está instalado. Ejecuta la celda del Paso 1.")


<a id="paso-03"></a>

## Paso 3. Definir el visor HTML para mostrar registros sin Pandas

Esta función presenta listas de diccionarios como tablas HTML.


In [34]:
def mostrar_tabla(registros, limite=30):
    """Muestra una lista de diccionarios como tabla HTML."""
    registros = list(registros)
    if not registros:
        display(HTML("<p><strong>No hay registros.</strong></p>"))
        return

    visibles = registros[:limite]
    columnas = list(visibles[0].keys())
    cabecera = "".join(
        f"<th>{escape(str(columna))}</th>"
        for columna in columnas
    )
    filas = []
    for registro in visibles:
        celdas = "".join(
            f"<td>{escape(str(registro.get(columna, '')))}</td>"
            for columna in columnas
        )
        filas.append(f"<tr>{celdas}</tr>")

    html = f"""
    <p>Mostrando {len(visibles)} de {len(registros)} registros.</p>
    <div style="overflow:auto; max-height:500px">
      <table style="border-collapse:collapse">
        <thead><tr>{cabecera}</tr></thead>
        <tbody>{''.join(filas)}</tbody>
      </table>
    </div>
    <style>
      th, td {{border:1px solid #ccc; padding:5px 8px; text-align:left;}}
      th {{background:#1f4e78; color:white; position:sticky; top:0;}}
    </style>
    """
    display(HTML(html))


<a id="paso-04"></a>

## Paso 4. Localizar y validar la ruta del archivo Excel original


In [35]:
candidatas = [
    Path("BDD/BaseDatosProyecto.xlsx"),
    Path("Excel/BDD/BaseDatosProyecto.xlsx"),
    Path.cwd() / "BDD" / "BaseDatosProyecto.xlsx",
    Path.cwd() / "Excel" / "BDD" / "BaseDatosProyecto.xlsx",
]

RUTA_ORIGINAL = next((ruta.resolve() for ruta in candidatas if ruta.exists()), None)
if RUTA_ORIGINAL is None:
    raise FileNotFoundError("No se encontró Excel/BDD/BaseDatosProyecto.xlsx")

print(f"Original: {RUTA_ORIGINAL}")


Original: /home/tvt/MEGA/EjerciciosPython/Excel/BDD/BaseDatosProyecto.xlsx


<a id="paso-05"></a>

## Paso 5. Crear una copia de trabajo para proteger el archivo original

Cada ejecución completa parte nuevamente del original. Así los ejercicios CRUD no dañan la base de datos inicial.


In [36]:
RUTA_TRABAJO = RUTA_ORIGINAL.with_name("BaseDatosProyecto_trabajo.xlsx")
copy2(RUTA_ORIGINAL, RUTA_TRABAJO)
print(f"Copia creada: {RUTA_TRABAJO}")


Copia creada: /home/tvt/MEGA/EjerciciosPython/Excel/BDD/BaseDatosProyecto_trabajo.xlsx


<a id="paso-06"></a>

## Paso 6. Cargar el libro y seleccionar la hoja de gestión de pedidos


In [37]:
libro = load_workbook(RUTA_TRABAJO)
print("Hojas disponibles:")
for posicion, nombre in enumerate(libro.sheetnames, start=1):
    print(f"{posicion}. {nombre}")

NOMBRE_HOJA = "G. Pedidos"
if NOMBRE_HOJA not in libro.sheetnames:
    NOMBRE_HOJA = libro.sheetnames[0]

hoja = libro[NOMBRE_HOJA]
print(f"\nHoja seleccionada: {hoja.title}")
print(f"Filas físicas: {hoja.max_row}")
print(f"Columnas físicas: {hoja.max_column}")


Hojas disponibles:
1. G. Pedidos

Hoja seleccionada: G. Pedidos
Filas físicas: 2156
Columnas físicas: 9


<a id="paso-07"></a>

## Paso 7. Inspeccionar las cabeceras originales y detectar caracteres ocultos

Se usa `repr()` para hacer visibles saltos de línea y espacios repetidos.


In [38]:
cabeceras_originales = [
    hoja.cell(row=1, column=columna).value
    for columna in range(1, hoja.max_column + 1)
]

for numero, cabecera in enumerate(cabeceras_originales, start=1):
    print(f"{numero:>2}. {repr(cabecera)}")


 1. 'Cliente'
 2. 'Artículo'
 3. 'Fecha Pedido'
 4. 'Fecha Entrega'
 5. 'País Destino'
 6. 'Ciudad \n destino'
 7. 'Precio  \nunidad'
 8. 'Cantidad'
 9. 'Descuento'


<a id="paso-08"></a>

## Paso 8. Limpiar saltos de línea y espacios repetidos en las cabeceras

Los saltos de línea y espacios repetidos se sustituyen por un espacio normal.


In [39]:
def normalizar_cabecera(valor):
    return re.sub(r"\s+", " ", str(valor)).strip()

for columna in range(1, hoja.max_column + 1):
    celda = hoja.cell(row=1, column=columna)
    celda.value = normalizar_cabecera(celda.value)

cabeceras = [
    hoja.cell(row=1, column=columna).value
    for columna in range(1, hoja.max_column + 1)
]
print(cabeceras)


['Cliente', 'Artículo', 'Fecha Pedido', 'Fecha Entrega', 'País Destino', 'Ciudad destino', 'Precio unidad', 'Cantidad', 'Descuento']


<a id="paso-09"></a>

## Paso 9. Verificar que existan las nueve columnas obligatorias


In [40]:
CABECERAS_BASE = [
    "Cliente",
    "Artículo",
    "Fecha Pedido",
    "Fecha Entrega",
    "País Destino",
    "Ciudad destino",
    "Precio unidad",
    "Cantidad",
    "Descuento",
]

faltantes = [cabecera for cabecera in CABECERAS_BASE if cabecera not in cabeceras]
print("Cabeceras faltantes:", faltantes or "Ninguna")
if faltantes:
    raise ValueError(f"Faltan cabeceras obligatorias: {faltantes}")


Cabeceras faltantes: Ninguna


<a id="paso-10"></a>

## Paso 10. Crear el mapa dinámico entre nombres y números de columna

Trabajar por nombre evita depender de letras fijas como G, H o I.


In [41]:
def obtener_indices(hoja_excel):
    return {
        normalizar_cabecera(hoja_excel.cell(1, columna).value): columna
        for columna in range(1, hoja_excel.max_column + 1)
    }

indices = obtener_indices(hoja)
for nombre, numero in indices.items():
    print(f"{nombre}: columna {get_column_letter(numero)}")


Cliente: columna A
Artículo: columna B
Fecha Pedido: columna C
Fecha Entrega: columna D
País Destino: columna E
Ciudad destino: columna F
Precio unidad: columna G
Cantidad: columna H
Descuento: columna I


<a id="paso-11"></a>

## Paso 11. Añadir o reutilizar la columna Total al final de la hoja

Si `Total` no existe, se añade en la última columna. Si ya existe, se reutiliza y se recalculan sus valores.


In [42]:
TOTAL_CREADO = "Total" not in indices

if TOTAL_CREADO:
    columna_total = hoja.max_column + 1
    hoja.cell(row=1, column=columna_total, value="Total")
    hoja.cell(row=1, column=columna_total).font = Font(bold=True, color="FFFFFF")
    hoja.cell(row=1, column=columna_total).fill = PatternFill("solid", fgColor="1F4E78")
    hoja.cell(row=1, column=columna_total).alignment = Alignment(horizontal="center")
else:
    columna_total = indices["Total"]

indices = obtener_indices(hoja)
print("La columna Total fue creada." if TOTAL_CREADO else "La columna Total ya existía.")
print(f"Total está en la columna {get_column_letter(columna_total)}")


La columna Total fue creada.
Total está en la columna J


<a id="paso-12"></a>

## Paso 12. Definir las validaciones y la fórmula del importe total

La fórmula aplicada mediante Python es:

`Total = Precio unidad × Cantidad × (1 − Descuento)`


In [43]:
def numero_seguro(valor, nombre_campo):
    if isinstance(valor, bool) or not isinstance(valor, (int, float)):
        raise ValueError(f"{nombre_campo} debe ser numérico. Valor recibido: {valor!r}")
    return float(valor)


def calcular_total(precio, cantidad, descuento):
    precio = numero_seguro(precio, "Precio unidad")
    cantidad = numero_seguro(cantidad, "Cantidad")
    descuento = numero_seguro(descuento, "Descuento")

    if precio < 0:
        raise ValueError("Precio unidad no puede ser negativo.")
    if cantidad <= 0:
        raise ValueError("Cantidad debe ser mayor que cero.")
    if not 0 <= descuento <= 1:
        raise ValueError("Descuento debe estar entre 0 y 1.")

    return round(precio * cantidad * (1 - descuento), 2)

print("Ejemplo:", calcular_total(20, 5, 0.10))


Ejemplo: 90.0


<a id="paso-13"></a>

## Paso 13. Calcular y guardar el Total de todos los pedidos existentes

Los resultados se guardan como números para que puedan filtrarse y resumirse inmediatamente con Python. `openpyxl` no calcula fórmulas de Excel por sí solo.


In [44]:
indices = obtener_indices(hoja)
errores_total = []
filas_calculadas = 0

for fila in range(2, hoja.max_row + 1):
    try:
        total = calcular_total(
            hoja.cell(fila, indices["Precio unidad"]).value,
            hoja.cell(fila, indices["Cantidad"]).value,
            hoja.cell(fila, indices["Descuento"]).value,
        )
        celda_total = hoja.cell(fila, indices["Total"])
        celda_total.value = total
        celda_total.number_format = '#,##0.00'
        filas_calculadas += 1
    except ValueError as error:
        errores_total.append({"Fila Excel": fila, "Error": str(error)})

print(f"Totales calculados: {filas_calculadas}")
print(f"Errores: {len(errores_total)}")
mostrar_tabla(errores_total)


Totales calculados: 2155
Errores: 0


<a id="paso-14"></a>

## Paso 14. Guardar la estructura y los totales en la copia de trabajo


In [45]:
libro.save(RUTA_TRABAJO)
print(f"Cambios guardados en: {RUTA_TRABAJO}")


Cambios guardados en: /home/tvt/MEGA/EjerciciosPython/Excel/BDD/BaseDatosProyecto_trabajo.xlsx


<a id="paso-15"></a>

## Paso 15. Convertir las filas de Excel en registros identificados por fila física

La clave especial `_fila_excel` permite identificar la fila física durante el CRUD.


In [46]:
def leer_registros(hoja_excel):
    nombres = [
        normalizar_cabecera(hoja_excel.cell(1, columna).value)
        for columna in range(1, hoja_excel.max_column + 1)
    ]
    registros = []
    for numero_fila, valores in enumerate(
        hoja_excel.iter_rows(min_row=2, values_only=True), start=2
    ):
        registro = dict(zip(nombres, valores))
        registro = {"_fila_excel": numero_fila, **registro}
        registros.append(registro)
    return registros

registros = leer_registros(hoja)
print(f"Registros leídos: {len(registros)}")
mostrar_tabla(registros, limite=2000)


Registros leídos: 2155


_fila_excel,Cliente,Artículo,Fecha Pedido,Fecha Entrega,País Destino,Ciudad destino,Precio unidad,Cantidad,Descuento,Total
2,Alfreds Futterkiste,Arenque salado,2019-08-25 00:00:00,2019-09-22 00:00:00,Alemania,Berlin,18.615938591742193,20,0.25,279.24
3,Alfreds Futterkiste,Caracoles de Borgoña,2020-04-09 00:00:00,2020-05-07 00:00:00,Alemania,Berlin,20.555063684204768,40,0.05,781.09
4,Alfreds Futterkiste,Col fermentada Rössle,2019-08-25 00:00:00,2019-09-22 00:00:00,Alemania,Berlin,70.74056664862034,15,0.25,795.83
5,Alfreds Futterkiste,Col fermentada Rössle,2020-03-16 00:00:00,2020-04-27 00:00:00,Alemania,Berlin,70.74056664862034,2,0,141.48
6,Alfreds Futterkiste,Crema de queso Fløtemys,2020-04-09 00:00:00,2020-05-07 00:00:00,Alemania,Berlin,33.35348628851696,20,0,667.07
7,Alfreds Futterkiste,Licor Cloudberry,2019-10-13 00:00:00,2019-11-24 00:00:00,Alemania,Berlin,27.92390788761329,15,0,418.86
8,Alfreds Futterkiste,Licor verde Chartreuse,2019-08-25 00:00:00,2019-09-22 00:00:00,Alemania,Berlin,27.92390788761329,21,0.25,439.8
9,Alfreds Futterkiste,Mermelada de grosellas de la abuela,2020-03-16 00:00:00,2020-04-27 00:00:00,Alemania,Berlin,38.78320539946291,16,0.05,589.5
10,Alfreds Futterkiste,Raclet de queso Courdavault,2020-01-15 00:00:00,2020-02-12 00:00:00,Alemania,Berlin,85.3230518788184,15,0,1279.85
11,Alfreds Futterkiste,Salsa verde original Frankfurter,2020-01-15 00:00:00,2020-02-12 00:00:00,Alemania,Berlin,20.167266807720708,2,0.2,32.27


<a id="paso-16"></a>

## Paso 16. Construir una función reutilizable de filtros combinados


In [47]:
def filtrar_registros(
    registros_origen,
    cliente=None,
    articulo=None,
    pais=None,
    ciudad=None,
    cantidad_min=None,
    descuento_min=None,
    total_min=None,
):
    cliente = cliente.strip() if isinstance(cliente, str) else cliente
    articulo = articulo.strip() if isinstance(articulo, str) else articulo
    pais = pais.strip() if isinstance(pais, str) else pais
    ciudad = ciudad.strip() if isinstance(ciudad, str) else ciudad
    resultado = []

    for registro in registros_origen:
        if cliente and cliente.lower() not in str(registro["Cliente"]).lower():
            continue
        if articulo and articulo.lower() not in str(registro["Artículo"]).lower():
            continue
        if pais and pais.lower() not in str(registro["País Destino"]).lower():
            continue
        if ciudad and ciudad.lower() not in str(registro["Ciudad destino"]).lower():
            continue
        if cantidad_min is not None and registro["Cantidad"] < cantidad_min:
            continue
        if descuento_min is not None and registro["Descuento"] < descuento_min:
            continue
        if total_min is not None and registro["Total"] < total_min:
            continue
        resultado.append(registro)

    return resultado


<a id="paso-17"></a>

## Paso 17. Ejecutar una consulta de ejemplo y revisar sus coincidencias


In [48]:
consulta = filtrar_registros(
    registros,
    pais="Alemania",
    cantidad_min=20,
    descuento_min=0.05,
)

print(f"Coincidencias: {len(consulta)}")
mostrar_tabla(consulta, limite=20)


Coincidencias: 101


_fila_excel,Cliente,Artículo,Fecha Pedido,Fecha Entrega,País Destino,Ciudad destino,Precio unidad,Cantidad,Descuento,Total
2,Alfreds Futterkiste,Arenque salado,2019-08-25 00:00:00,2019-09-22 00:00:00,Alemania,Berlin,18.615938591742193,20,0.25,279.24
3,Alfreds Futterkiste,Caracoles de Borgoña,2020-04-09 00:00:00,2020-05-07 00:00:00,Alemania,Berlin,20.555063684204768,40,0.05,781.09
8,Alfreds Futterkiste,Licor verde Chartreuse,2019-08-25 00:00:00,2019-09-22 00:00:00,Alemania,Berlin,27.92390788761329,21,0.25,439.8
323,Die Wandernde Kuh,Bollos de Sir Rodney's,2019-04-22 00:00:00,2019-06-03 00:00:00,Alemania,Stuttgart,15.513282159785163,40,0.2,496.43
329,Die Wandernde Kuh,Cerveza tibetana Barley,2019-08-14 00:00:00,2019-09-11 00:00:00,Alemania,Stuttgart,29.47523610359181,30,0.05,840.04
333,Die Wandernde Kuh,Pan de centeno crujiente estilo Gustaf's,2019-09-01 00:00:00,2019-09-29 00:00:00,Alemania,Stuttgart,32.577892535548834,20,0.25,488.67
338,Die Wandernde Kuh,Queso de cabra,2019-08-14 00:00:00,2019-09-11 00:00:00,Alemania,Stuttgart,3.878390894967423,20,0.05,73.69
342,Die Wandernde Kuh,Queso Gudbrandsdals,2019-08-21 00:00:00,2019-09-18 00:00:00,Alemania,Stuttgart,55.84781577522658,20,0.25,837.72
343,Die Wandernde Kuh,Queso Manchego La Pastora,2020-04-23 00:00:00,2020-05-21 00:00:00,Alemania,Stuttgart,58.95047220718362,20,0.05,1120.06
344,Die Wandernde Kuh,Queso Mascarpone Fabioli,2019-04-22 00:00:00,2019-06-03 00:00:00,Alemania,Stuttgart,49.64250291131252,50,0.2,1985.7


<a id="paso-18"></a>

## Paso 18. Explorar pedidos con un panel de filtros opcionales


In [49]:
def convertir_numero_opcional(texto, nombre):
    """Convierte un filtro numérico o devuelve None si está vacío."""
    texto = str(texto).strip()
    if texto == "":
        return None
    try:
        return float(texto)
    except ValueError as error:
        raise ValueError(f"{nombre} debe ser numérico o quedar vacío.") from error


def mostrar_panel_filtros():
    if not WIDGETS_DISPONIBLES:
        print("Instala ipywidgets para activar el panel.")
        return

    aviso = widgets.HTML(
        "<b>Todos los campos son opcionales.</b> "
        "Déjalos vacíos para mostrar todos los registros."
    )
    cliente = widgets.Text(description="Cliente:", placeholder="Todos")
    articulo = widgets.Text(description="Artículo:", placeholder="Todos")
    pais = widgets.Text(description="País:", placeholder="Todos")
    ciudad = widgets.Text(description="Ciudad:", placeholder="Todas")
    cantidad = widgets.Text(description="Cant. mín.:", placeholder="Sin mínimo")
    total = widgets.Text(description="Total mín.:", placeholder="Sin mínimo")
    aplicar = widgets.Button(description="Aplicar filtros", button_style="primary")
    limpiar = widgets.Button(description="Mostrar todos")
    salida = widgets.Output()

    def ejecutar(_):
        with salida:
            clear_output()
            try:
                cantidad_min = convertir_numero_opcional(
                    cantidad.value, "Cantidad mínima"
                )
                total_min = convertir_numero_opcional(
                    total.value, "Total mínimo"
                )
                actuales = leer_registros(hoja)
                filtrados = filtrar_registros(
                    actuales,
                    cliente=cliente.value.strip() or None,
                    articulo=articulo.value.strip() or None,
                    pais=pais.value.strip() or None,
                    ciudad=ciudad.value.strip() or None,
                    cantidad_min=cantidad_min,
                    total_min=total_min,
                )
                print(f"Coincidencias: {len(filtrados)}")
                mostrar_tabla(filtrados, limite=100)
            except ValueError as error:
                print(f"Error de filtro: {error}")

    def restablecer(_):
        cliente.value = ""
        articulo.value = ""
        pais.value = ""
        ciudad.value = ""
        cantidad.value = ""
        total.value = ""
        ejecutar(None)

    aplicar.on_click(ejecutar)
    limpiar.on_click(restablecer)
    display(widgets.VBox([
        aviso,
        widgets.HBox([cliente, articulo]),
        widgets.HBox([pais, ciudad]),
        widgets.HBox([cantidad, total]),
        widgets.HBox([aplicar, limpiar]),
        salida,
    ]))
    ejecutar(None)


mostrar_panel_filtros()


<a id="paso-19"></a>

## Paso 19. Validar textos, fechas y números antes de ejecutar el CRUD


In [50]:
def convertir_fecha(valor, campo):
    if isinstance(valor, datetime):
        return valor
    if isinstance(valor, date):
        return datetime.combine(valor, datetime.min.time())
    try:
        return datetime.strptime(str(valor), "%Y-%m-%d")
    except ValueError as error:
        raise ValueError(f"{campo} debe usar el formato AAAA-MM-DD.") from error


def validar_pedido(pedido):
    errores = []
    for campo in ["Cliente", "Artículo", "País Destino", "Ciudad destino"]:
        if not str(pedido.get(campo, "")).strip():
            errores.append(f"{campo} es obligatorio.")

    try:
        pedido_fecha = convertir_fecha(pedido.get("Fecha Pedido"), "Fecha Pedido")
        entrega_fecha = convertir_fecha(pedido.get("Fecha Entrega"), "Fecha Entrega")
        if entrega_fecha < pedido_fecha:
            errores.append("Fecha Entrega no puede ser anterior a Fecha Pedido.")
    except ValueError as error:
        errores.append(str(error))

    try:
        calcular_total(
            float(pedido.get("Precio unidad")),
            int(pedido.get("Cantidad")),
            float(pedido.get("Descuento")),
        )
    except (TypeError, ValueError) as error:
        errores.append(str(error))

    return errores


<a id="paso-20"></a>

## Paso 20. Implementar la creación y consulta con Total automático


In [51]:
def crear_pedido(hoja_excel, pedido):
    errores = validar_pedido(pedido)
    if errores:
        raise ValueError("\n".join(errores))

    indices_locales = obtener_indices(hoja_excel)
    nueva_fila = hoja_excel.max_row + 1
    valores = dict(pedido)
    valores["Fecha Pedido"] = convertir_fecha(valores["Fecha Pedido"], "Fecha Pedido")
    valores["Fecha Entrega"] = convertir_fecha(valores["Fecha Entrega"], "Fecha Entrega")
    valores["Precio unidad"] = float(valores["Precio unidad"])
    valores["Cantidad"] = int(valores["Cantidad"])
    valores["Descuento"] = float(valores["Descuento"])
    valores["Total"] = calcular_total(
        valores["Precio unidad"], valores["Cantidad"], valores["Descuento"]
    )

    for campo, columna in indices_locales.items():
        hoja_excel.cell(nueva_fila, columna, valores.get(campo))

    hoja_excel.cell(nueva_fila, indices_locales["Fecha Pedido"]).number_format = "dd/mm/yyyy"
    hoja_excel.cell(nueva_fila, indices_locales["Fecha Entrega"]).number_format = "dd/mm/yyyy"
    hoja_excel.cell(nueva_fila, indices_locales["Total"]).number_format = '#,##0.00'
    return nueva_fila


def leer_pedido(hoja_excel, fila_excel):
    fila_excel = int(fila_excel)
    if fila_excel < 2 or fila_excel > hoja_excel.max_row:
        raise KeyError(f"No existe la fila Excel {fila_excel}.")
    nombres = list(obtener_indices(hoja_excel))
    valores = [hoja_excel.cell(fila_excel, c).value for c in range(1, hoja_excel.max_column + 1)]
    return {"_fila_excel": fila_excel, **dict(zip(nombres, valores))}


<a id="paso-21"></a>

## Paso 21. Implementar la actualización protegida y la eliminación de pedidos


In [52]:
CAMPOS_ACTUALIZABLES = ["Precio unidad", "Cantidad", "Descuento"]


def actualizar_pedido(hoja_excel, fila_excel, cambios):
    campos_no_permitidos = [
        campo for campo in cambios
        if campo not in CAMPOS_ACTUALIZABLES
    ]
    if campos_no_permitidos:
        raise KeyError(
            "Solo se pueden actualizar Precio unidad, Cantidad y Descuento. "
            f"Campos no permitidos: {campos_no_permitidos}"
        )

    actual = leer_pedido(hoja_excel, fila_excel)
    candidato = {campo: actual[campo] for campo in CABECERAS_BASE}
    candidato.update(cambios)
    errores = validar_pedido(candidato)
    if errores:
        raise ValueError("\n".join(errores))

    indices_locales = obtener_indices(hoja_excel)
    for campo, valor in cambios.items():
        if campo == "Cantidad":
            valor = int(valor)
        elif campo in ["Precio unidad", "Descuento"]:
            valor = float(valor)
        hoja_excel.cell(int(fila_excel), indices_locales[campo], valor)

    total = calcular_total(
        hoja_excel.cell(int(fila_excel), indices_locales["Precio unidad"]).value,
        hoja_excel.cell(int(fila_excel), indices_locales["Cantidad"]).value,
        hoja_excel.cell(int(fila_excel), indices_locales["Descuento"]).value,
    )
    hoja_excel.cell(int(fila_excel), indices_locales["Total"], total)
    return leer_pedido(hoja_excel, fila_excel)


def eliminar_pedido(hoja_excel, fila_excel, confirmar=False):
    if not confirmar:
        raise ValueError("Debes confirmar la eliminación.")
    eliminado = leer_pedido(hoja_excel, fila_excel)
    hoja_excel.delete_rows(int(fila_excel), 1)
    return eliminado


<a id="paso-22"></a>

## Paso 22. Probar las operaciones CRUD mediante ejemplos manuales

Están comentados para evitar cambios accidentales.


In [53]:
# CREAR
# nueva_fila = crear_pedido(hoja, {
#     "Cliente": "Cliente nuevo",
#     "Artículo": "Artículo nuevo",
#     "Fecha Pedido": "2026-09-30",
#     "Fecha Entrega": "2026-10-05",
#     "País Destino": "España",
#     "Ciudad destino": "Madrid",
#     "Precio unidad": 25.50,
#     "Cantidad": 4,
#     "Descuento": 0.10,
# })
# mostrar_tabla([leer_pedido(hoja, nueva_fila)])

# ACTUALIZAR
# mostrar_tabla([actualizar_pedido(hoja, 2, {"Cantidad": 30})])

# ELIMINAR
# mostrar_tabla([eliminar_pedido(hoja, 2, confirmar=True)])


<a id="paso-23"></a>

## Paso 23. Gestionar pedidos desde una consola con búsqueda previa

Ejecuta la celda y selecciona una opción. En la actualización solo aparecen tres campos editables:

1. `Precio unidad`
2. `Cantidad`
3. `Descuento`

`Total` nunca se solicita ni se modifica directamente. Se recalcula automáticamente después de crear o actualizar un pedido.


In [54]:
def pedir_texto_consola(nombre):
    while True:
        valor = input(f"{nombre}: ").strip()
        if valor:
            return valor
        print(f"Error: {nombre} es obligatorio.")


def pedir_fecha_consola(nombre):
    """Solicita y valida una fecha antes de continuar al siguiente campo."""
    while True:
        valor = input(f"{nombre} (AAAA-MM-DD): ").strip()
        try:
            fecha = convertir_fecha(valor, nombre)
            print(f"Fecha válida: {fecha:%d/%m/%Y}")
            return fecha
        except ValueError as error:
            print(f"Error inmediato: {error}")


def pedir_fecha_entrega_consola(fecha_pedido):
    """Valida formato y orden cronológico antes de continuar."""
    while True:
        fecha_entrega = pedir_fecha_consola("Fecha Entrega")
        if fecha_entrega < fecha_pedido:
            print(
                "Error inmediato: Fecha Entrega no puede ser "
                "anterior a Fecha Pedido."
            )
            continue
        return fecha_entrega


def pedir_precio_consola():
    while True:
        valor = input("Precio unidad: ").strip()
        try:
            numero = float(valor)
            if numero < 0:
                raise ValueError("El precio no puede ser negativo.")
            return numero
        except ValueError as error:
            print(f"Error inmediato: {error}")


def pedir_cantidad_consola():
    while True:
        valor = input("Cantidad: ").strip()
        try:
            numero = int(valor)
            if numero <= 0:
                raise ValueError("La cantidad debe ser mayor que cero.")
            return numero
        except ValueError as error:
            print(f"Error inmediato: {error}")


def pedir_descuento_consola():
    while True:
        valor = input("Descuento entre 0 y 1: ").strip()
        try:
            numero = float(valor)
            if not 0 <= numero <= 1:
                raise ValueError("El descuento debe estar entre 0 y 1.")
            return numero
        except ValueError as error:
            print(f"Error inmediato: {error}")


def pedir_valor_actualizacion(campo):
    if campo == "Precio unidad":
        return pedir_precio_consola()
    if campo == "Cantidad":
        return pedir_cantidad_consola()
    return pedir_descuento_consola()


def buscar_fila_actualizacion_consola():
    """Busca pedidos y obliga a elegir una fila de las coincidencias."""
    print("\nBÚSQUEDA PREVIA DEL PEDIDO")
    print("Deja vacíos los filtros que no quieras utilizar.")
    coincidencias = filtrar_registros(
        leer_registros(hoja),
        cliente=input("Cliente: "),
        articulo=input("Artículo: "),
        pais=input("País: "),
        ciudad=input("Ciudad: "),
    )

    if not coincidencias:
        print("No se encontraron pedidos con esos filtros.")
        return None

    mostrar_tabla(coincidencias, limite=50)
    filas_permitidas = {
        registro["_fila_excel"] for registro in coincidencias
    }

    while True:
        valor = input(
            "Fila Excel que quieres actualizar "
            "(Enter para cancelar): "
        ).strip()
        if valor == "":
            print("Actualización cancelada.")
            return None
        try:
            fila = int(valor)
        except ValueError:
            print("La fila debe ser un número entero.")
            continue

        if fila not in filas_permitidas:
            print("La fila no pertenece a los resultados mostrados.")
            continue

        print("Pedido seleccionado:")
        mostrar_tabla([leer_pedido(hoja, fila)])
        return fila


def menu_crud():
    while True:
        print("\n" + "=" * 52)
        print("        GESTIÓN DE PEDIDOS CON OPENPYXL")
        print("=" * 52)
        print("1. Crear pedido")
        print("2. Consultar pedido por fila Excel")
        print("3. Actualizar precio, cantidad o descuento")
        print("4. Eliminar pedido")
        print("5. Guardar cambios")
        print("6. Salir")
        print("=" * 52)

        opcion = input("Selecciona una opción (1-6): ").strip()

        try:
            if opcion == "1":
                print("\nIntroduce los datos. Cada valor se valida inmediatamente.")
                fecha_pedido = pedir_fecha_consola("Fecha Pedido")
                fecha_entrega = pedir_fecha_entrega_consola(fecha_pedido)

                pedido = {
                    "Cliente": pedir_texto_consola("Cliente"),
                    "Artículo": pedir_texto_consola("Artículo"),
                    "Fecha Pedido": fecha_pedido,
                    "Fecha Entrega": fecha_entrega,
                    "País Destino": pedir_texto_consola("País Destino"),
                    "Ciudad destino": pedir_texto_consola("Ciudad destino"),
                    "Precio unidad": pedir_precio_consola(),
                    "Cantidad": pedir_cantidad_consola(),
                    "Descuento": pedir_descuento_consola(),
                }
                nueva_fila = crear_pedido(hoja, pedido)
                print(f"Pedido creado en la fila {nueva_fila}.")
                print("Total calculado automáticamente:")
                mostrar_tabla([leer_pedido(hoja, nueva_fila)])

            elif opcion == "2":
                fila = int(input("Fila Excel que quieres consultar: "))
                mostrar_tabla([leer_pedido(hoja, fila)])

            elif opcion == "3":
                fila = buscar_fila_actualizacion_consola()
                if fila is None:
                    continue
                print("\nCampos permitidos:")
                print("1. Precio unidad")
                print("2. Cantidad")
                print("3. Descuento")
                campo_elegido = input("Selecciona un campo (1-3): ").strip()
                campos = {
                    "1": "Precio unidad",
                    "2": "Cantidad",
                    "3": "Descuento",
                }
                if campo_elegido not in campos:
                    print("Opción de campo inválida.")
                    continue

                campo = campos[campo_elegido]
                nuevo_valor = pedir_valor_actualizacion(campo)
                actualizado = actualizar_pedido(
                    hoja,
                    fila,
                    {campo: nuevo_valor},
                )
                print("Pedido actualizado. Total recalculado automáticamente:")
                mostrar_tabla([actualizado])

            elif opcion == "4":
                fila = int(input("Fila Excel que quieres eliminar: "))
                mostrar_tabla([leer_pedido(hoja, fila)])
                confirmacion = input(
                    "Escribe ELIMINAR para confirmar: "
                ).strip()
                if confirmacion == "ELIMINAR":
                    eliminar_pedido(hoja, fila, confirmar=True)
                    print("Pedido eliminado.")
                else:
                    print("Eliminación cancelada.")

            elif opcion == "5":
                libro.save(RUTA_TRABAJO)
                print(f"Cambios guardados en {RUTA_TRABAJO}")

            elif opcion == "6":
                print("Consola CRUD finalizada.")
                break

            else:
                print("Opción inválida. Selecciona un número del 1 al 6.")

        except (ValueError, KeyError) as error:
            print(f"Error: {error}")


# Quita el símbolo # para abrir la consola cuando quieras probarla:
# menu_crud()


<a id="paso-24"></a>

## Paso 24. Gestionar pedidos con ipywidgets y validación en tiempo real

Este visor ofrece las mismas operaciones en cuatro pestañas. La pestaña **Crear** valida mientras escribes:

- Las fechas se comprueban al seleccionarlas.
- La entrega no puede ser anterior al pedido.
- El botón queda desactivado mientras exista un error.
- El total se muestra como vista previa y nunca es editable.

La pestaña **Actualizar** solo permite precio, cantidad y descuento, y muestra el total proyectado antes de guardar.


In [55]:
def crear_visor_crud():
    if not WIDGETS_DISPONIBLES:
        print("Instala ipywidgets para activar el visor.")
        return None

    # ---------- Pestaña Crear ----------
    crear_campos = {
        "Cliente": widgets.Text(description="Cliente:"),
        "Artículo": widgets.Text(description="Artículo:"),
        "Fecha Pedido": widgets.DatePicker(description="Pedido:"),
        "Fecha Entrega": widgets.DatePicker(description="Entrega:"),
        "País Destino": widgets.Text(description="País:"),
        "Ciudad destino": widgets.Text(description="Ciudad:"),
        "Precio unidad": widgets.FloatText(description="Precio:", value=0),
        "Cantidad": widgets.IntText(description="Cantidad:", value=1),
        "Descuento": widgets.FloatSlider(
            description="Descuento:", min=0, max=1, step=0.05, value=0
        ),
    }
    crear_estado = widgets.HTML()
    crear_total = widgets.HTML(value="<b>Total calculado: pendiente</b>")
    crear_boton = widgets.Button(
        description="Crear pedido",
        button_style="success",
        disabled=True,
    )
    crear_salida = widgets.Output()

    def datos_creacion():
        return {
            nombre: control.value
            for nombre, control in crear_campos.items()
        }

    def validar_creacion(_=None):
        pedido = datos_creacion()
        errores = validar_pedido(pedido)

        if errores:
            crear_estado.value = (
                "<span style='color:#b91c1c'><b>Revisar:</b> "
                + escape(" | ".join(errores))
                + "</span>"
            )
            crear_total.value = "<b>Total calculado: pendiente</b>"
            crear_boton.disabled = True
            return

        total = calcular_total(
            pedido["Precio unidad"],
            pedido["Cantidad"],
            pedido["Descuento"],
        )
        crear_estado.value = (
            "<span style='color:#15803d'><b>Datos válidos.</b></span>"
        )
        crear_total.value = (
            f"<b>Total calculado automáticamente: {total:,.2f}</b>"
        )
        crear_boton.disabled = False

    def ejecutar_creacion(_):
        with crear_salida:
            clear_output()
            validar_creacion()
            if crear_boton.disabled:
                print("Corrige los errores antes de crear.")
                return
            try:
                fila_nueva = crear_pedido(hoja, datos_creacion())
                print(f"Pedido creado en la fila {fila_nueva}.")
                mostrar_tabla([leer_pedido(hoja, fila_nueva)])
            except (ValueError, KeyError) as error:
                print(f"Error: {error}")

    for control in crear_campos.values():
        control.observe(validar_creacion, names="value")
    crear_boton.on_click(ejecutar_creacion)
    validar_creacion()

    pestaña_crear = widgets.VBox([
        *crear_campos.values(),
        crear_estado,
        crear_total,
        crear_boton,
        crear_salida,
    ])

    # ---------- Pestaña Consultar ----------
    consultar_fila = widgets.IntText(description="Fila Excel:", value=2)
    consultar_boton = widgets.Button(
        description="Consultar", button_style="info"
    )
    consultar_salida = widgets.Output()

    def ejecutar_consulta(_):
        with consultar_salida:
            clear_output()
            try:
                mostrar_tabla([leer_pedido(hoja, consultar_fila.value)])
            except KeyError as error:
                print(f"Error: {error}")

    consultar_boton.on_click(ejecutar_consulta)
    pestaña_consultar = widgets.VBox([
        consultar_fila,
        consultar_boton,
        consultar_salida,
    ])

    # ---------- Pestaña Actualizar ----------
    buscar_cliente = widgets.Text(description="Cliente:", placeholder="Opcional")
    buscar_articulo = widgets.Text(description="Artículo:", placeholder="Opcional")
    buscar_pais = widgets.Text(description="País:", placeholder="Opcional")
    buscar_ciudad = widgets.Text(description="Ciudad:", placeholder="Opcional")
    buscar_boton = widgets.Button(
        description="Buscar pedidos",
        button_style="info",
    )
    buscar_salida = widgets.Output()

    actualizar_fila = widgets.Dropdown(
        description="Pedido:",
        options=[],
        disabled=True,
    )
    actualizar_campo = widgets.Dropdown(
        description="Campo:",
        options=CAMPOS_ACTUALIZABLES,
    )
    actualizar_valor = widgets.Text(description="Nuevo valor:")
    actualizar_estado = widgets.HTML(
        value="<span>Primero realiza una búsqueda.</span>"
    )
    actualizar_boton = widgets.Button(
        description="Actualizar",
        button_style="warning",
        disabled=True,
    )
    actualizar_salida = widgets.Output()

    def ejecutar_busqueda_actualizacion(_=None):
        with buscar_salida:
            clear_output()
            resultados = filtrar_registros(
                leer_registros(hoja),
                cliente=buscar_cliente.value,
                articulo=buscar_articulo.value,
                pais=buscar_pais.value,
                ciudad=buscar_ciudad.value,
            )
            print(f"Coincidencias: {len(resultados)}")
            mostrar_tabla(resultados, limite=50)

        opciones = [
            (
                f"Fila {r['_fila_excel']} - "
                f"{r['Cliente']} - {r['Artículo']}",
                r["_fila_excel"],
            )
            for r in resultados
        ]
        actualizar_fila.options = opciones
        actualizar_fila.disabled = not bool(opciones)
        if opciones:
            actualizar_fila.value = opciones[0][1]
            actualizar_estado.value = (
                "<span>Selecciona el pedido, el campo y el nuevo valor.</span>"
            )
        else:
            actualizar_estado.value = (
                "<span style='color:#b91c1c'>"
                "No hay pedidos para seleccionar.</span>"
            )
        validar_actualizacion()

    def convertir_actualizacion(campo, texto):
        texto = texto.strip()
        if campo == "Cantidad":
            valor = int(texto)
            if valor <= 0:
                raise ValueError("Cantidad debe ser mayor que cero.")
            return valor
        valor = float(texto)
        if campo == "Precio unidad" and valor < 0:
            raise ValueError("Precio unidad no puede ser negativo.")
        if campo == "Descuento" and not 0 <= valor <= 1:
            raise ValueError("Descuento debe estar entre 0 y 1.")
        return valor

    def validar_actualizacion(_=None):
        try:
            if actualizar_fila.value is None:
                raise ValueError("Selecciona un pedido.")
            actual = leer_pedido(hoja, actualizar_fila.value)
            campo = actualizar_campo.value
            valor = convertir_actualizacion(campo, actualizar_valor.value)
            precio = valor if campo == "Precio unidad" else actual["Precio unidad"]
            cantidad = valor if campo == "Cantidad" else actual["Cantidad"]
            descuento = valor if campo == "Descuento" else actual["Descuento"]
            total_nuevo = calcular_total(precio, cantidad, descuento)
            actualizar_estado.value = (
                "<span style='color:#15803d'><b>Valor válido.</b> "
                f"Total proyectado: {total_nuevo:,.2f}</span>"
            )
            actualizar_boton.disabled = False
        except (ValueError, KeyError, TypeError):
            actualizar_estado.value = (
                "<span style='color:#b91c1c'>"
                "Busca un pedido e introduce un valor válido.</span>"
            )
            actualizar_boton.disabled = True

    def ejecutar_actualizacion(_):
        with actualizar_salida:
            clear_output()
            validar_actualizacion()
            if actualizar_boton.disabled:
                print("Corrige la selección o el valor.")
                return
            try:
                valor = convertir_actualizacion(
                    actualizar_campo.value,
                    actualizar_valor.value,
                )
                resultado = actualizar_pedido(
                    hoja,
                    actualizar_fila.value,
                    {actualizar_campo.value: valor},
                )
                print("Pedido actualizado. Total recalculado:")
                mostrar_tabla([resultado])
                ejecutar_busqueda_actualizacion()
            except (ValueError, KeyError) as error:
                print(f"Error: {error}")

    buscar_boton.on_click(ejecutar_busqueda_actualizacion)
    actualizar_fila.observe(validar_actualizacion, names="value")
    actualizar_campo.observe(validar_actualizacion, names="value")
    actualizar_valor.observe(validar_actualizacion, names="value")
    actualizar_boton.on_click(ejecutar_actualizacion)

    pestaña_actualizar = widgets.VBox([
        widgets.HTML("<b>1. Busca el pedido que quieres actualizar</b>"),
        widgets.HBox([buscar_cliente, buscar_articulo]),
        widgets.HBox([buscar_pais, buscar_ciudad]),
        buscar_boton,
        buscar_salida,
        widgets.HTML("<b>2. Selecciona una coincidencia</b>"),
        actualizar_fila,
        widgets.HTML("<b>3. Indica el cambio</b>"),
        actualizar_campo,
        actualizar_valor,
        actualizar_estado,
        actualizar_boton,
        actualizar_salida,
    ])

    # ---------- Pestaña Eliminar ----------
    eliminar_fila = widgets.IntText(description="Fila Excel:", value=2)
    eliminar_ver = widgets.Button(description="Vista previa")
    eliminar_confirmar = widgets.Checkbox(
        description="Confirmo la eliminación"
    )
    eliminar_boton = widgets.Button(
        description="Eliminar",
        button_style="danger",
        disabled=True,
    )
    eliminar_salida = widgets.Output()

    def ver_eliminacion(_):
        with eliminar_salida:
            clear_output()
            try:
                mostrar_tabla([leer_pedido(hoja, eliminar_fila.value)])
            except KeyError as error:
                print(f"Error: {error}")

    def cambiar_confirmacion(cambio):
        eliminar_boton.disabled = not cambio["new"]

    def ejecutar_eliminacion(_):
        with eliminar_salida:
            clear_output()
            try:
                eliminado = eliminar_pedido(
                    hoja,
                    eliminar_fila.value,
                    confirmar=eliminar_confirmar.value,
                )
                print("Pedido eliminado:")
                mostrar_tabla([eliminado])
                eliminar_confirmar.value = False
            except (ValueError, KeyError) as error:
                print(f"Error: {error}")

    eliminar_ver.on_click(ver_eliminacion)
    eliminar_confirmar.observe(cambiar_confirmacion, names="value")
    eliminar_boton.on_click(ejecutar_eliminacion)

    pestaña_eliminar = widgets.VBox([
        eliminar_fila,
        eliminar_ver,
        eliminar_confirmar,
        eliminar_boton,
        eliminar_salida,
    ])

    pestañas = widgets.Tab(children=[
        pestaña_crear,
        pestaña_consultar,
        pestaña_actualizar,
        pestaña_eliminar,
    ])
    for indice, titulo in enumerate(
        ["Crear", "Consultar", "Actualizar", "Eliminar"]
    ):
        pestañas.set_title(indice, titulo)

    display(pestañas)
    return pestañas


visor_crud = crear_visor_crud()


<a id="paso-25"></a>

## Paso 25. Agrupar pedidos por país y conciliar cantidades y totales


In [56]:
registros_actuales = leer_registros(hoja)
resumen_paises = defaultdict(lambda: {"Pedidos": 0, "Cantidad": 0, "Total": 0.0})

for registro in registros_actuales:
    pais = registro["País Destino"]
    resumen_paises[pais]["Pedidos"] += 1
    resumen_paises[pais]["Cantidad"] += registro["Cantidad"]
    resumen_paises[pais]["Total"] += registro["Total"]

resumen_lista = [
    {
        "País Destino": pais,
        "Pedidos": valores["Pedidos"],
        "Cantidad": valores["Cantidad"],
        "Total": round(valores["Total"], 2),
    }
    for pais, valores in resumen_paises.items()
]
resumen_lista.sort(key=lambda r: r["Total"], reverse=True)
mostrar_tabla(resumen_lista, limite=20)


País Destino,Pedidos,Cantidad,Total
Estados Unidos,354,9335,381230.6
Alemania,328,9231,357498.36
Austria,125,5167,198575.97
Brasil,203,4247,165877.01
Francia,181,3227,125530.91
Reino Unido,134,2727,91130.17
Venezuela,118,2936,88131.94
Suecia,98,2250,84893.54
Canadá,75,1984,77870.97
Irlanda,55,1684,77535.24


<a id="paso-26"></a>

## Paso 26. Crear la hoja Resumen y el gráfico de totales por país

Si la hoja ya existe, se reemplaza para evitar duplicados al repetir el paso.


In [57]:
if "Resumen" in libro.sheetnames:
    libro.remove(libro["Resumen"])

hoja_resumen = libro.create_sheet("Resumen", 0)
hoja_resumen.append(["País Destino", "Pedidos", "Cantidad", "Total"])

for registro in resumen_lista:
    hoja_resumen.append([
        registro["País Destino"],
        registro["Pedidos"],
        registro["Cantidad"],
        registro["Total"],
    ])

for celda in hoja_resumen[1]:
    celda.font = Font(bold=True, color="FFFFFF")
    celda.fill = PatternFill("solid", fgColor="1F4E78")
    celda.alignment = Alignment(horizontal="center")

for fila in range(2, hoja_resumen.max_row + 1):
    hoja_resumen.cell(fila, 4).number_format = '#,##0.00'

for columna, ancho in {"A": 22, "B": 12, "C": 14, "D": 16}.items():
    hoja_resumen.column_dimensions[columna].width = ancho

cantidad_grafico = min(10, len(resumen_lista))
grafico = BarChart()
grafico.title = "Total por país"
grafico.y_axis.title = "Total"
grafico.x_axis.title = "País"
datos_grafico = Reference(hoja_resumen, min_col=4, min_row=1, max_row=cantidad_grafico + 1)
categorias_grafico = Reference(hoja_resumen, min_col=1, min_row=2, max_row=cantidad_grafico + 1)
grafico.add_data(datos_grafico, titles_from_data=True)
grafico.set_categories(categorias_grafico)
grafico.height = 8
grafico.width = 15
hoja_resumen.add_chart(grafico, "F2")
print("Hoja Resumen y gráfico creados.")


Hoja Resumen y gráfico creados.


<a id="paso-27"></a>

## Paso 27. Guardar en Excel todos los cambios realizados en memoria


In [58]:
libro.save(RUTA_TRABAJO)
print(f"Libro guardado: {RUTA_TRABAJO}")


Libro guardado: /home/tvt/MEGA/EjerciciosPython/Excel/BDD/BaseDatosProyecto_trabajo.xlsx


<a id="paso-28"></a>

## Paso 28. Reabrir el archivo y ejecutar las comprobaciones básicas


In [59]:
libro_verificacion = load_workbook(RUTA_TRABAJO, data_only=True)
hoja_verificacion = libro_verificacion[NOMBRE_HOJA]
indices_verificacion = obtener_indices(hoja_verificacion)

totales_verificados = [
    hoja_verificacion.cell(fila, indices_verificacion["Total"]).value
    for fila in range(2, hoja_verificacion.max_row + 1)
]

COMPROBACIONES_BASICAS = {
    "Archivo de trabajo guardado": RUTA_TRABAJO.exists(),
    "Columna Total presente": "Total" in indices_verificacion,
    "Totales guardados como números": all(
        isinstance(valor, (int, float))
        for valor in totales_verificados
    ),
    "Hoja Resumen presente": "Resumen" in libro_verificacion.sheetnames,
}

lineas = [
    f"- {'✅' if correcto else '❌'} **{nombre}**"
    for nombre, correcto in COMPROBACIONES_BASICAS.items()
]
display(Markdown(
    "### Comprobaciones básicas\n\n" + "\n".join(lineas)
))


### Comprobaciones básicas

- ✅ **Archivo de trabajo guardado**
- ✅ **Columna Total presente**
- ✅ **Totales guardados como números**
- ✅ **Hoja Resumen presente**

<a id="paso-29"></a>

## Paso 29. Mostrar el resultado consolidado final en formato Markdown

Esta última celda presenta el resultado consolidado del proceso.


In [60]:
registros_finales = hoja_verificacion.max_row - 1
total_general_final = round(sum(totales_verificados), 2)
comprobaciones_correctas = sum(COMPROBACIONES_BASICAS.values())
comprobaciones_totales = len(COMPROBACIONES_BASICAS)

display(Markdown(f"""
## Resultado final

- **Archivo original:** `{RUTA_ORIGINAL.name}`
- **Archivo generado:** `{RUTA_TRABAJO.name}`
- **Hoja de pedidos:** `{NOMBRE_HOJA}`
- **Registros finales:** {registros_finales:,}
- **Total general calculado:** {total_general_final:,.2f}
- **Comprobaciones básicas:** {comprobaciones_correctas}/{comprobaciones_totales}
- **Campos actualizables:** Precio unidad, Cantidad y Descuento
- **Campo Total:** protegido y calculado automáticamente
- **Interfaces disponibles:** consola e ipywidgets

El archivo original permanece intacto. Los cambios están guardados en la copia de trabajo.
"""))



## Resultado final

- **Archivo original:** `BaseDatosProyecto.xlsx`
- **Archivo generado:** `BaseDatosProyecto_trabajo.xlsx`
- **Hoja de pedidos:** `G. Pedidos`
- **Registros finales:** 2,155
- **Total general calculado:** 1,963,911.99
- **Comprobaciones básicas:** 4/4
- **Campos actualizables:** Precio unidad, Cantidad y Descuento
- **Campo Total:** protegido y calculado automáticamente
- **Interfaces disponibles:** consola e ipywidgets

El archivo original permanece intacto. Los cambios están guardados en la copia de trabajo.
